# Import

In [1]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [2]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Dependency
* DGIDB_hypergraph
* DDBC

# Prep

## Loading variables

In [3]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [4]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [5]:
# Loading result graph and communities
with open(f"{DISEASE_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{DISEASE_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{DISEASE_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [6]:
for c in communities:
    print(len(c))

3256
2905
2836
2670
2252
2045
1764
1730
717
706
422
409
41
34
34
28
27
24
22
14
8
7
5
4
3
2
2
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


## Helpful functions (big object, drop NAN)

In [7]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [8]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [9]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['84272', '5813', '55884', '64327', '4154', '23648', '23139', '51663', '9019', '9991', '57222', '54765', '9980', '113419', '4820', '9522', '7267', '55900', '6651', '9852', '10807', '55795', '10180', '157567', '81572', '23065', '55186', '54978', '11179', '81688', '9202', '56257', '57621', '5911', '9213', '55233', '10771', '55858', '9747', '104472715', '56951', '9898', '54788', '5412', '7259', '91404', '57180', '57409', '90355', '51108', '222658', '6397', '9993', '55206', '55556', '84726', '51290', '64756', '23484', '23271', '387263', '55608', '440026', '51123', '11079', '23613', '55754', '23392', '79647', '55041', '7095', '91304', '56987', '6666', '10314', '79073', '8780', '54726', '9779', '55031', '55325', '754', '8073', '83941', '27315', '10724', '136319', '26225', '124565', '23029', '64089', '10116', '9325', '6738', '7716', '64418', '9240', '10521', '57798', '10668', '22877', '25843', '88455', '8545', '25957', '116224', '114908', '57035', '9736', '152006', '11333', '285527', '80213'

In [10]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{DISEASE_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['83', '100', '118', '132', '143', '159', '166', '205', '228', '267', '271', '292', '310', '353', '373', '421', '439', '440', '473', '475', '520', '523', '526', '527', '528', '529', '533', '534', '550', '573', '631', '663', '711', '734', '745', '754', '770', '811', '813', '819', '821', '833', '889', '892', '900', '955', '989', '997', '1018', '1039', '1054', '1059', '1068', '1080', '1105', '1182', '1185', '1192', '1198', '1203', '1209', '1266', '1389', '1408', '1411', '1413', '1429', '1454', '1455', '1456', '1488', '1538', '1603', '1650', '1654', '1656', '1657', '1723', '1731', '1797', '1801', '1819', '1820', '1822', '1859', '1877', '1939', '1951', '1979', '1982', '1983', '1984', '1994', '2011', '2023', '2030', '2063', '2077', '2118', '2119', '2186', '2195', '2218', '2286', '2288', '2310', '2314', '2332', '2339', '2504', '2539', '2582', '2585', '2589', '2597', '2630', '2665', '2673', '2762', '2764', '2765', '2794', '2802', '2842', '2923', '2969', '2971', '2987', '3005', '3006', '3077',

## NCBI to HGNC

In [11]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [12]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [13]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [14]:
print(len(COMMUNITIES_HGNC))

12


In [15]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 2 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 1 NaN entries
Community 5: dropped 1 NaN entries
Community 6: dropped 1 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 4 NaN entries
Community 9: dropped 1 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries

Total dropped across all communities: 10
Community 0: dropped 5 NaN entries
Community 1: dropped 5 NaN entries
Community 2: dropped 4 NaN entries
Community 3: dropped 5 NaN entries
Community 4: dropped 2 NaN entries
Community 5: dropped 7 NaN entries
Community 6: dropped 3 NaN entries
Community 7: dropped 1 NaN entries
Community 8: dropped 5 NaN entries
Community 9: dropped 2 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Commun

In [16]:
with open(f"{DISEASE_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{DISEASE_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [17]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

12
49


# Categoization Prep

### GO-slim

In [18]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [19]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [20]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [21]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [22]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [23]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [24]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [25]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [26]:
TERM_SCORE_CAP = 1e-5
PERCENTAGE = 0.1

In [27]:
def enrichment(communities,
               term_score_cap,
               percentage, 
               db,
               term_to_category):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=db,
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["Category"] = filtered["Term"].apply(lambda term: term_to_category(term))

        # Get empty count
        empty_count = (filtered["Category"].apply(len) == 0).sum()
        
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Adjusted P-value'], ascending=True)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "genes_involved": involved,
            "n_involved": len(involved),
            "n_not_involved": len(not_involved)
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

### GO

In [28]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    rows = []
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        
        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["id"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["id"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # compute genes involved in enrichment
        community_set = set(community)
        hit_genes = set(";".join(filtered["Genes"].dropna()).split(";"))

        involved = sorted(community_set & hit_genes)
        not_involved = sorted(community_set - hit_genes)

        rows.append({
            "community": i,
            "n_genes": len(community_set),
            "n_involved": len(involved),
            "n_not_involved": len(not_involved),
        })
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
        
    community_coverage_df = pd.DataFrame(rows)
    
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,community_coverage_df

In [29]:
term = "Nuclear Pore Organization (GO:0006999)"
print(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))

{'GO:0009987'}


In [30]:
go_important_terms, go_community_coverage = enrichment(COMMUNITIES_HGNC,
                                                       TERM_SCORE_CAP,
                                                       PERCENTAGE,
                                                       ['GO_Biological_Process_2023',
                                                        'GO_Molecular_Function_2023',
                                                        'GO_Cellular_Component_2023'],
                                                       lambda term: [go[id].name for id in list(get_go_ancestors_at_depth(get_goid(term), depth=1, include_relations=("is_a", "part_of")))])

Size of community: 1120
Number of filtered terms: 14
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_54124\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
2148,0,Ubiquitin-Protein Transferase Activity (GO:0004842),80/412,2.129653e-20,[catalytic activity]
2526,0,cullin-RING Ubiquitin Ligase Complex (GO:0031461),46/174,6.459549e-17,[protein-containing complex]
0,0,Protein Ubiquitination (GO:0016567),77/434,2.324584e-16,[cellular process]
2149,0,Ubiquitin-Like Protein Ligase Activity (GO:0061659),61/319,3.574281e-15,[catalytic activity]
2150,0,Ubiquitin Protein Ligase Activity (GO:0061630),59/311,1.190994e-14,[catalytic activity]
2151,0,Ubiquitin-Like Protein Transferase Activity (GO:0019787),49/240,1.838966e-13,[catalytic activity]
1,0,Protein Modification By Small Protein Conjugation (GO:0032446),64/364,2.482763e-13,[cellular process]
2,0,Ubiquitin-Dependent Protein Catabolic Process (GO:0006511),60/367,4.286487e-11,[cellular process]
3,0,Modification-Dependent Protein Catabolic Process (GO:0019941),40/192,2.023429e-10,[cellular process]
4,0,Protein Polyubiquitination (GO:0000209),42/226,2.288618e-09,[cellular process]


Size of community: 1129
Number of filtered terms: 46
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
2160,1,Sequence-Specific DNA Binding (GO:0043565),188/717,3.200258e-73,[binding]
2161,1,Double-Stranded DNA Binding (GO:0003690),178/650,1.311967e-72,[binding]
2162,1,Sequence-Specific Double-Stranded DNA Binding (GO:1990837),186/715,3.066593e-72,[binding]
2163,1,RNA Polymerase II Transcription Regulatory Region Sequence-Specific DNA Binding (GO:0000977),221/1225,1.735447e-55,[binding]
2164,1,RNA Polymerase II Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000978),190/1122,7.275046e-43,[binding]
2165,1,G Protein-Coupled Receptor Activity (GO:0004930),83/250,5.819468e-40,[molecular transducer activity]
2166,1,Cis-Regulatory Region Sequence-Specific DNA Binding (GO:0000987),179/1098,5.434616e-38,[binding]
0,1,Regulation Of Transcription By RNA Polymerase II (GO:0006357),252/2028,4.947830e-32,[biological regulation]
1,1,Adenylate Cyclase-Modulating G Protein-Coupled Receptor Signaling Pathway (GO:0007188),60/163,2.728433e-30,"[biological regulation, cellular process]"
2167,1,G Protein-Coupled Peptide Receptor Activity (GO:0008528),39/77,8.638396e-27,[molecular transducer activity]


...

Size of community: 238
Number of filtered terms: 90
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Golgi Vesicle Transport (GO:0048193),84/197,5.895122e-110,"[localization, cellular process]"
1,10,Endoplasmic Reticulum To Golgi Vesicle-Mediated Transport (GO:0006888),59/115,8.078741e-82,"[localization, cellular process]"
2,10,Vesicle-Mediated Transport (GO:0016192),70/411,5.743346e-59,"[localization, cellular process]"
3,10,Protein Transport (GO:0015031),64/313,5.743346e-59,[localization]
4,10,Intracellular Protein Transport (GO:0006886),59/325,7.058900e-51,"[localization, cellular process]"
838,10,Golgi Membrane (GO:0000139),58/427,2.099015e-42,[cellular anatomical structure]
5,10,Protein Localization (GO:0008104),54/351,2.580620e-42,[localization]
6,10,"Retrograde Transport, Endosome To Golgi (GO:0042147)",32/94,2.297380e-36,"[localization, cellular process]"
840,10,trans-Golgi Network (GO:0005802),42/241,4.597349e-35,[cellular anatomical structure]
7,10,Endosomal Transport (GO:0016197),38/180,1.129190e-34,"[localization, cellular process]"


Size of community: 217
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
30,11,Olfactory Receptor Activity (GO:0004984),174/362,1.574092e-279,[molecular transducer activity]
0,11,Sensory Perception Of Smell (GO:0007608),112/230,6.660332e-167,[multicellular organismal process]
1,11,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),69/141,2.527752e-99,[response to stimulus]
2,11,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),68/139,5.700298e-98,[response to stimulus]
3,11,Sensory Perception Of Chemical Stimulus (GO:0007606),53/110,3.358297e-75,[multicellular organismal process]


12 out of 12 communities had significant GO terms.


In [31]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1120,Ubiquitin-Protein Transferase Activity (GO:000...,80/412,2.129653e-20,[catalytic activity],GO_Molecular_Function_2023,5.634003e-23,0.0,0.0,4.297498,220.163544,RNF10;UBE2D4;RNF13;PPP1R11;RNF14;LTN1;UBE3A;UB...,0.194175
1,0,1120,cullin-RING Ubiquitin Ligase Complex (GO:0031461),46/174,6.459549e-17,[protein-containing complex],GO_Cellular_Component_2023,3.166446e-19,0.0,0.0,6.274674,267.279200,CUL9;CUL7;TMEM183A;PEF1;PDCD6;DCAF8;KLHL13;DCA...,0.264368
2,0,1120,Protein Ubiquitination (GO:0016567),77/434,2.324584e-16,[cellular process],GO_Biological_Process_2023,1.082208e-19,0.0,0.0,3.830448,167.276080,RNF10;UBE2D4;RNF13;RNF14;FBXO28;UBE3A;UBE3B;DD...,0.177419
3,0,1120,Ubiquitin-Like Protein Ligase Activity (GO:006...,61/319,3.574281e-15,[catalytic activity],GO_Molecular_Function_2023,1.891154e-17,0.0,0.0,4.157579,160.094888,ZNF451;RNF10;PPP1R11;RNF13;RNF14;LTN1;UBE3A;UB...,0.191223
4,0,1120,Ubiquitin Protein Ligase Activity (GO:0061630),59/311,1.190994e-14,[catalytic activity],GO_Molecular_Function_2023,9.452335e-17,0.0,0.0,4.110573,151.670610,RNF10;PPP1R11;RNF13;RNF14;LTN1;UBE3A;UBE3B;HER...,0.189711
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1151,11,217,Olfactory Receptor Activity (GO:0004984),174/362,1.574092e-279,[molecular transducer activity],GO_Molecular_Function_2023,5.246974e-280,0.0,0.0,421.762741,271221.352567,OR1C1;OR5K4;OR52N1;OR2M5;OR2M4;OR2T12;OR2T10;O...,0.480663
1152,11,217,Sensory Perception Of Smell (GO:0007608),112/230,6.660332e-167,[multicellular organismal process],GO_Biological_Process_2023,2.220111e-168,0.0,0.0,177.762712,68622.937530,OR1C1;OR2M5;OR2M4;OR2T12;OR2T10;OR10AC1;OR2T11...,0.486957
1153,11,217,Detection Of Chemical Stimulus Involved In Sen...,69/141,2.527752e-99,[response to stimulus],GO_Biological_Process_2023,1.685168e-100,0.0,0.0,127.633164,29322.014847,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR13H1;...,0.489362
1154,11,217,Detection Of Chemical Stimulus Involved In Sen...,68/139,5.700298e-98,[response to stimulus],GO_Biological_Process_2023,5.700298e-99,0.0,0.0,126.705360,28662.704343,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR13H1;...,0.489209


In [32]:
go_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1120,"[AKIRIN2, AKTIP, APPBP2, AREL1, ARIH1, ARIH2, ...",173,947
1,1,1129,"[ABCC8, ADCYAP1, ADCYAP1R1, ADGRB1, ADGRB2, AD...",541,588
2,2,1201,"[AARSD1, AATF, ABT1, ACIN1, ADAT1, ADNP, AGBL5...",851,350
3,3,918,"[ABLIM1, AHNAK, AIF1L, AUTS2, BIN1, CALD1, CDC...",40,878
4,4,981,"[AASS, ABCA1, ABCA12, ABCA2, ABCA3, ABCA5, ABC...",487,494
5,5,913,"[ACOD1, ADAM8, ADGRE5, ADGRG3, AIF1, ANPEP, AN...",476,437
6,6,712,"[ABI3BP, ADAM12, ADAM15, ADAM19, ADAM28, ADAM2...",413,299
7,7,754,"[ABI1, ABI2, ACTA1, ACTB, ACTG1, ACTN1, ACTN4,...",654,100
8,8,359,"[ANKS3, ARL3, ARMC2, B9D1, BBIP1, BBOF1, BBS4,...",108,251
9,9,336,"[ACTL6A, ACTL6B, ACTRT1, AICDA, AKAP8L, ARID1A...",231,105


### KEGG

In [33]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [34]:
kegg_important_terms, kegg_community_coverage = enrichment(communities = COMMUNITIES_HGNC,
                                 term_score_cap = TERM_SCORE_CAP,
                                 percentage = PERCENTAGE,
                                 db = ['KEGG_2021_Human'],
                                 term_to_category = lambda term: get_kegg_level2(name_to_id.get(term.lower())))

Size of community: 1129
Number of filtered terms: 1
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_54124\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,Neuroactive ligand-receptor interaction,105/341,5.061953e-47,[Signaling molecules and interaction]


Size of community: 1201
Number of filtered terms: 5
Number of unmapped terms: 1


,Community Index,Term,Overlap,Adjusted P-value,Category
0,2,Spliceosome,88/150,3.804545e-66,[Transcription]
1,2,Ribosome biogenesis in eukaryotes,45/108,3.121871e-25,[Translation]
2,2,mRNA surveillance pathway,34/98,3.387777e-16,[Translation]
3,2,RNA transport,47/186,3.422042e-16,[]
4,2,Cell cycle,36/124,1.479491e-14,[Cell growth and death]


...

Size of community: 238
Number of filtered terms: 2
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,SNARE interactions in vesicular transport,18/33,4.751985e-25,"[Folding, sorting and degradation]"
1,10,Endocytosis,26/252,8.838411e-16,[Transport and catabolism]


Size of community: 217
Number of filtered terms: 1
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Olfactory transduction,210/440,0.0,[Sensory system]


9 out of 12 communities had significant GO terms.


In [35]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,1,1129,Neuroactive ligand-receptor interaction,105/341,5.061953e-47,[Signaling molecules and interaction],KEGG_2021_Human,3.864087e-49,0.0,0.0,8.096676,902.576455,OXTR;VIPR1;VIPR2;NPY2R;PMCH;EDNRA;EDNRB;PTGDR;...,0.307918
1,2,1201,Spliceosome,88/150,3.804545e-66,[Transcription],KEGG_2021_Human,4.227272e-68,0.0,0.0,23.894386,3706.837160,TCERG1;RBM25;DDX46;EIF4A3;DDX42;HNRNPU;PQBP1;S...,0.586667
2,2,1201,Ribosome biogenesis in eukaryotes,45/108,3.121871e-25,[Translation],KEGG_2021_Human,6.937491e-27,0.0,0.0,11.576866,697.307720,POP5;RBM28;POP7;NXT1;POP1;RPP30;WDR3;HEATR1;FC...,0.416667
3,2,1201,mRNA surveillance pathway,34/98,3.387777e-16,[Translation],KEGG_2021_Human,1.129259e-17,0.0,0.0,8.528679,332.809408,DAZAP1;NXT1;RBM8A;CSTF3;EIF4A3;CSTF2;CSTF2T;CA...,0.346939
4,2,1201,RNA transport,47/186,3.422042e-16,[],KEGG_2021_Human,1.520907e-17,0.0,0.0,5.467501,211.727013,POP5;POP7;NXT1;RBM8A;POP1;GEMIN2;RPP30;EIF4A3;...,0.252688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,9,336,Neutrophil extracellular trap formation,24/189,1.085169e-13,[Immune system],KEGG_2021_Human,1.391242e-14,0.0,0.0,9.090443,290.039622,H2AX;H4C7;H2AZ2;H2AC8;H2AC6;H2BC5;H2AB3;H2AC1;...,0.126984
203,9,336,Hepatocellular carcinoma,17/168,2.437205e-08,[Cancer: specific types],KEGG_2021_Human,3.749547e-09,0.0,0.0,6.886608,133.611418,SMARCE1;SMARCD1;SMARCC1;PBRM1;SMARCD2;SMARCB1;...,0.101190
204,10,238,SNARE interactions in vesicular transport,18/33,4.751985e-25,"[Folding, sorting and degradation]",KEGG_2021_Human,1.055997e-26,0.0,0.0,107.710909,6442.483245,STX16;GOSR2;STX19;GOSR1;STX18;STX7;USE1;BET1L;...,0.545455
205,10,238,Endocytosis,26/252,8.838411e-16,[Transport and catabolism],KEGG_2021_Human,3.928183e-17,0.0,0.0,10.601436,400.477403,ARF3;ARF4;ARF1;RAB5B;RAB5C;CYTH3;CYTH2;CYTH4;C...,0.103175


In [36]:
kegg_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1120,[],0,1120
1,1,1129,"[ADCYAP1, ADCYAP1R1, ADORA2A, ADORA2B, AGTR1, ...",105,1024
2,2,1201,"[ACIN1, BCAS2, BMS1, BUB1, BUB1B, BUB3, BUD31,...",216,985
3,3,918,[],0,918
4,4,981,"[AACS, ABCA1, ABCA12, ABCA2, ABCA3, ABCA5, ABC...",328,653
5,5,913,"[ANPEP, ANTXR2, B2M, BCL2A1, BCL3, BIRC3, CARD...",310,603
6,6,712,"[CD44, CELA2A, CELA3A, CELA3B, CHAD, COL10A1, ...",79,633
7,7,754,"[ABI1, ABI2, ACTB, ACTG1, ACTN1, ACTN4, ACTR2,...",580,174
8,8,359,[],0,359
9,9,336,"[ACTL6A, ACTL6B, ARID1A, ARID1B, ARID2, ASH1L,...",63,273


### Reactome

In [37]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [38]:
reactome_important_terms, reactome_community_coverage = enrichment(COMMUNITIES_HGNC,
                                      TERM_SCORE_CAP,
                                      PERCENTAGE,
                                      ['Reactome_2022'],
                                      lambda term: reactome_level1.get(term.split(" ")[-1],[]))

Size of community: 1120
Number of filtered terms: 4
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_54124\1924290263.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
0,0,Antigen Processing: Ubiquitination And Proteasome Degradation R-HSA-983168,86/307,1.312370e-34,[Immune System]
1,0,Class I MHC Mediated Antigen Processing And Presentation R-HSA-983169,86/378,1.315687e-27,[Immune System]
2,0,Neddylation R-HSA-8951664,61/237,1.948360e-22,[Metabolism of proteins]
3,0,Adaptive Immune System R-HSA-1280218,87/733,1.576378e-09,[Immune System]


Size of community: 1129
Number of filtered terms: 18
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,1,GPCR Ligand Binding R-HSA-500792,122/458,5.782614e-47,[Signal Transduction]
1,1,Class A/1 (Rhodopsin-like Receptors) R-HSA-373076,97/327,1.311139e-41,[Signal Transduction]
2,1,Signaling By GPCR R-HSA-372790,142/689,3.861356e-41,[Signal Transduction]
3,1,GPCR Downstream Signaling R-HSA-388396,126/619,8.978973e-36,[Signal Transduction]
4,1,Peptide Ligand-Binding Receptors R-HSA-375276,65/196,4.848030e-31,[Signal Transduction]
5,1,G Alpha (S) Signaling Events R-HSA-418555,45/153,5.447765e-19,[Signal Transduction]
6,1,ADORA2B Mediated Anti-Inflammatory Cytokine Production R-HSA-9660821,39/131,1.121106e-16,[Disease]
7,1,Voltage Gated Potassium Channels R-HSA-1296072,23/43,2.077257e-16,[Neuronal System]
8,1,Potassium Channels R-HSA-1296071,32/102,1.791878e-14,[Neuronal System]
9,1,Anti-inflammatory Response Favoring Leishmania Infection R-HSA-9662851,39/165,4.095459e-13,[Disease]


...

Size of community: 238
Number of filtered terms: 29
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,10,Membrane Trafficking R-HSA-199991,219/599,0.000000e+00,[Vesicle-mediated transport]
1,10,Vesicle-mediated Transport R-HSA-5653656,219/637,3.718401e-317,[Vesicle-mediated transport]
2,10,Intra-Golgi And Retrograde Golgi-to-ER Traffic R-HSA-6811442,104/181,1.841866e-158,[Vesicle-mediated transport]
3,10,ER To Golgi Anterograde Transport R-HSA-199977,74/133,1.123960e-108,"[Metabolism of proteins, Vesicle-mediated transport]"
4,10,Rab Regulation Of Trafficking R-HSA-9007101,71/122,3.812225e-106,[Vesicle-mediated transport]
5,10,Transport To Golgi And Subsequent Modification R-HSA-948021,74/164,1.205416e-99,[Metabolism of proteins]
6,10,Golgi-to-ER Retrograde Transport R-HSA-8856688,60/112,4.970879e-86,[Vesicle-mediated transport]
7,10,Asparagine N-linked Glycosylation R-HSA-446203,74/282,8.627792e-79,[Metabolism of proteins]
8,10,RAB GEFs Exchange GTP For GDP On RABs R-HSA-8876198,52/89,4.497367e-77,[Vesicle-mediated transport]
9,10,COPI-dependent Golgi-to-ER Retrograde Traffic R-HSA-6811434,49/78,7.349606e-75,[Vesicle-mediated transport]


Size of community: 217
Number of filtered terms: 3
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Category
0,11,Olfactory Signaling Pathway R-HSA-381753,204/401,0.000000e+00,[Sensory Perception]
1,11,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,204/393,0.000000e+00,[Sensory Perception]
2,11,Sensory Perception R-HSA-9709957,204/616,5.984879e-305,[Sensory Perception]


11 out of 12 communities had significant GO terms.


In [39]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1120,Antigen Processing: Ubiquitination And Proteas...,86/307,1.312370e-34,[Immune System],Reactome_2022,4.621019e-37,0.0,0.0,7.022213,5.875137e+02,UBE2D4;RNF14;LTN1;UBE3A;UBE3B;FBXO21;HERC4;NPE...,0.280130
1,0,1120,Class I MHC Mediated Antigen Processing And Pr...,86/378,1.315687e-27,[Immune System],Reactome_2022,9.265402e-30,0.0,0.0,5.294534,3.539463e+02,UBE2D4;RNF14;LTN1;UBE3A;UBE3B;FBXO21;HERC4;NPE...,0.227513
2,0,1120,Neddylation R-HSA-8951664,61/237,1.948360e-22,[Metabolism of proteins],Reactome_2022,2.058127e-24,0.0,0.0,6.121470,3.338665e+02,DCAF8;DCAF5;FBXO21;DCAF4;DCAF6;DDA1;NUB1;UBXN7...,0.257384
3,0,1120,Adaptive Immune System R-HSA-1280218,87/733,1.576378e-09,[Immune System],Reactome_2022,2.220251e-11,0.0,0.0,2.377214,5.831501e+01,UBE2D4;RNF14;CLSTN1;LTN1;UBE3A;UBE3B;FBXO21;NP...,0.118690
4,1,1129,GPCR Ligand Binding R-HSA-500792,122/458,5.782614e-47,[Signal Transduction],Reactome_2022,1.762992e-49,0.0,0.0,6.683188,7.502524e+02,OXTR;VIPR1;VIPR2;GPR68;NPY2R;PMCH;IHH;EDNRA;ED...,0.266376
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,10,238,Golgi Cisternae Pericentriolar Stack Reorganiz...,5/14,2.396149e-06,[Cell Cycle],Reactome_2022,4.196644e-07,0.0,0.0,47.098236,6.915816e+02,GOLGA2;RAB1A;RAB1B;GORASP1;USO1,0.357143
546,10,238,"Antigen Presentation: Folding, Assembly, Pepti...",6/28,4.461827e-06,[Immune System],Reactome_2022,8.066580e-07,0.0,0.0,23.205329,3.255793e+02,SEC23A;SEC24B;SEC24A;SEC24D;SEC24C;SEC31A,0.214286
547,11,217,Olfactory Signaling Pathway R-HSA-381753,204/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,1560.149941,inf,OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;OR4Q3;OR10AC1...,0.508728
548,11,217,Expression And Translocation Of Olfactory Rece...,204/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,1626.852259,inf,OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;OR4Q3;OR10AC1...,0.519084


In [40]:
reactome_community_coverage

,community,n_genes,genes_involved,n_involved,n_not_involved
0,0,1120,"[AREL1, ARIH2, ASB1, ASB13, ASB6, ASB7, ASB9, ...",111,1009
1,1,1129,"[ABCC8, ACKR2, ACKR4, ADCYAP1, ADCYAP1R1, ADGR...",173,956
2,2,1201,"[ADAT1, AHCTF1, AIMP2, ARPP19, AURKA, AURKAIP1...",436,765
3,3,918,[],0,918
4,4,981,"[AACS, AASS, ABCA1, ABCA12, ABCA2, ABCA3, ABCA...",483,498
5,5,913,"[ADA2, ADAM8, ADGRE5, ADGRG3, ANPEP, ANXA1, AN...",392,521
6,6,712,"[ADAM12, ADAM15, ADAM19, ADAM9, ADAMTS1, ADAMT...",270,442
7,7,754,"[AAMP, ABHD17A, ABI1, ABI2, ABR, ACTG1, ACTN1,...",630,124
8,8,359,"[ARL3, B9D1, BBIP1, BBS4, CYS1, DYNLT2, DYNLT5...",23,336
9,9,336,"[ACTL6A, ACTL6B, AEBP2, ARID1A, ARID1B, ARID2,...",253,83


# Important Terms df

In [41]:
community_coverage_combined = go_community_coverage.copy()

community_coverage_combined["genes_involved"] = [
    set(a) | set(b) | set(c)
    for a, b, c in zip(go_community_coverage["genes_involved"], kegg_community_coverage["genes_involved"], reactome_community_coverage["genes_involved"])
]
community_coverage_combined["n_involved"] = community_coverage_combined["genes_involved"].apply(len)
community_coverage_combined["n_not_involved"] = community_coverage_combined["n_genes"] - community_coverage_combined["n_involved"]
community_coverage_combined["percentage_involved"] = community_coverage_combined["n_involved"] / community_coverage_combined["n_genes"]

In [42]:
community_coverage_combined

,community,n_genes,genes_involved,n_involved,n_not_involved,percentage_involved
0,0,1120,"{COMMD3, NEDD4, UBL4A, FBXO17, OTUD4, USP47, F...",194,926,0.173214
1,1,1129,"{SFRP1, VAX1, KCNG1, NXPH4, RGS6, SLITRK4, GRI...",573,556,0.507529
2,2,1201,"{MTRFR, RAD54B, MSH2, SNRPD1, SNRPD3, WDR76, S...",876,325,0.729392
3,3,918,"{TSPAN5, DSTN, AHNAK, MYO1B, WDR1, CDC42EP3, M...",40,878,0.043573
4,4,981,"{ALDH1A3, CIDEC, NUDT6, SLC38A4, MGST1, CBR4, ...",588,393,0.599388
5,5,913,"{IL18, CD244, SLAMF6, TNFSF8, CD1D, AIF1, IFNG...",549,364,0.601314
6,6,712,"{LAMA5, COL9A1, CAPN5, NPR3, TCHH, KRTAP24-1, ...",447,265,0.627809
7,7,754,"{PKN3, VRK3, AKAP13, MAP3K1, JMJD1C, CAPN2, MS...",738,16,0.978780
8,8,359,"{CFAP251, TEKT2, DNAAF11, ROPN1L, DNAAF10, CFA...",112,247,0.311978
9,9,336,"{EP400, BRD1, ELP3, H2AX, ZZZ3, ZNF454, ELP6, ...",304,32,0.904762


In [43]:
comm_to_involved_pct = dict(zip(community_coverage_combined["community"], community_coverage_combined["percentage_involved"]))

with open(DISEASE_FOLDER + "comm_to_involved_pct.json", "w") as f:
    json.dump(comm_to_involved_pct, f, indent=2)


In [44]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1120,Ubiquitin-Protein Transferase Activity (GO:000...,80/412,2.129653e-20,[catalytic activity],GO_Molecular_Function_2023,5.634003e-23,0.0,0.0,4.297498,2.201635e+02,RNF10;UBE2D4;RNF13;PPP1R11;RNF14;LTN1;UBE3A;UB...,0.194175
1366,0,1120,Adaptive Immune System R-HSA-1280218,87/733,1.576378e-09,[Immune System],Reactome_2022,2.220251e-11,0.0,0.0,2.377214,5.831501e+01,UBE2D4;RNF14;CLSTN1;LTN1;UBE3A;UBE3B;FBXO21;NP...,0.118690
1365,0,1120,Neddylation R-HSA-8951664,61/237,1.948360e-22,[Metabolism of proteins],Reactome_2022,2.058127e-24,0.0,0.0,6.121470,3.338665e+02,DCAF8;DCAF5;FBXO21;DCAF4;DCAF6;DDA1;NUB1;UBXN7...,0.257384
1363,0,1120,Antigen Processing: Ubiquitination And Proteas...,86/307,1.312370e-34,[Immune System],Reactome_2022,4.621019e-37,0.0,0.0,7.022213,5.875137e+02,UBE2D4;RNF14;LTN1;UBE3A;UBE3B;FBXO21;HERC4;NPE...,0.280130
13,0,1120,Cysteine-Type Deubiquitinase Activity (GO:0004...,21/98,4.122296e-06,[catalytic activity],GO_Molecular_Function_2023,8.724436e-08,0.0,0.0,4.666143,7.584606e+01,OTUD4;OTUB2;USP36;USP47;USP25;USP48;USP6;USP49...,0.214286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1152,11,217,Sensory Perception Of Smell (GO:0007608),112/230,6.660332e-167,[multicellular organismal process],GO_Biological_Process_2023,2.220111e-168,0.0,0.0,177.762712,6.862294e+04,OR1C1;OR2M5;OR2M4;OR2T12;OR2T10;OR10AC1;OR2T11...,0.486957
1151,11,217,Olfactory Receptor Activity (GO:0004984),174/362,1.574092e-279,[molecular transducer activity],GO_Molecular_Function_2023,5.246974e-280,0.0,0.0,421.762741,2.712214e+05,OR1C1;OR5K4;OR52N1;OR2M5;OR2M4;OR2T12;OR2T10;O...,0.480663
1911,11,217,Expression And Translocation Of Olfactory Rece...,204/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,1626.852259,inf,OR11H4;OR52N1;OR2M5;OR2M4;OR4K17;OR4Q3;OR10AC1...,0.519084
1153,11,217,Detection Of Chemical Stimulus Involved In Sen...,69/141,2.527752e-99,[response to stimulus],GO_Biological_Process_2023,1.685168e-100,0.0,0.0,127.633164,2.932201e+04,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR13H1;...,0.489362


## Remove Redundant Terms

In [45]:
TERM_SIZE_CAP = 100

In [46]:
def parse_genes(gene_str):
    # Enrichr "Genes" field is like "IL6;STAT3;JAK2"
    return set(re.split(r"[;, ]+", gene_str.strip()))

df = important_terms.sort_values("Adjusted P-value")

kept_rows = []
kept_gene_sets = []

for _, row in df.iterrows():
    genes = parse_genes(row["Genes"])
    if len(genes) == 0:
        continue
    too_similar = any(len(genes & g)/len(genes | g) > 0.35 for g in kept_gene_sets)
    if not too_similar and int(row['Overlap'].split("/")[1]) < TERM_SIZE_CAP:
        kept_rows.append(row)
        kept_gene_sets.append(genes)

important_terms_nonredundant = pd.DataFrame(kept_rows).sort_values(by=["Community Index", "Adjusted P-value"])

In [47]:
important_terms_nonredundant

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
11,0,1120,Ubiquitin Ligase-Substrate Adaptor Activity (G...,16/44,5.987088e-08,[molecular adaptor activity],GO_Molecular_Function_2023,7.919429e-10,0.0,0.0,9.757764,204.488891,KLHDC2;KLHDC3;FEM1A;KLHL25;PEF1;FEM1B;PDCD6;FB...,0.363636
13,0,1120,Cysteine-Type Deubiquitinase Activity (GO:0004...,21/98,4.122296e-06,[catalytic activity],GO_Molecular_Function_2023,8.724436e-08,0.0,0.0,4.666143,75.846065,OTUD4;OTUB2;USP36;USP47;USP25;USP48;USP6;USP49...,0.214286
23,1,1129,G Protein-Coupled Peptide Receptor Activity (G...,39/77,8.638396e-27,[molecular transducer activity],GO_Molecular_Function_2023,1.872823e-28,0.0,0.0,17.732665,1132.140880,OPRD1;OXTR;VIPR1;VIPR2;NPR1;MLNR;OPRL1;FPR3;GP...,0.506494
27,1,1129,Anterior/Posterior Pattern Specification (GO:0...,28/59,3.548246e-17,[multicellular organismal process],GO_Biological_Process_2023,8.213533e-20,0.0,0.0,15.455744,679.216871,GATA4;HHEX;SIX2;HOXA3;HOXC5;HOXC4;HES3;HOXC9;H...,0.474576
28,1,1129,Voltage-Gated Potassium Channel Complex (GO:00...,30/73,1.127232e-16,[protein-containing complex],GO_Cellular_Component_2023,6.515793e-19,0.0,0.0,11.952515,500.510224,DPP10;KCNG1;HCN4;KCNC1;LRRC38;KCNA1;KCNC4;KCNA...,0.410959
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,10,238,Exocytosis (GO:0006887),10/86,1.063367e-06,"[localization, cellular process]",GO_Biological_Process_2023,7.565454e-08,0.0,0.0,11.360803,186.284096,NSF;RAB10;SNAP47;VAMP7;TMED10;STX19;MIA3;SNAP2...,0.116279
1908,10,238,Golgi Cisternae Pericentriolar Stack Reorganiz...,5/14,2.396149e-06,[Cell Cycle],Reactome_2022,4.196644e-07,0.0,0.0,47.098236,691.581561,GOLGA2;RAB1A;RAB1B;GORASP1;USO1,0.357143
1146,10,238,Protein Localization To Endoplasmic Reticulum ...,4/6,3.833169e-06,[localization],GO_Biological_Process_2023,2.878665e-07,0.0,0.0,168.888889,2543.596563,SEC16B;SEC16A;GBF1;MIA3,0.666667
1909,10,238,"Antigen Presentation: Folding, Assembly, Pepti...",6/28,4.461827e-06,[Immune System],Reactome_2022,8.066580e-07,0.0,0.0,23.205329,325.579263,SEC23A;SEC24B;SEC24A;SEC24D;SEC24C;SEC31A,0.214286


## Save Important Terms

In [48]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [ ]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [ ]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [ ]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# twr3

In [ ]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# terms_with_recurrence

In [ ]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [ ]:
# terms_with_rec_merged

In [ ]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [ ]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))